# Review Summarization Pipeline

This notebook loads the Amazon Reviews dataset, filters reviews for a specific product category, and prepares text data for summarization using transformer models.

In [1]:
# Setup & Library Imports
import sys
import torch
import pandas as pd
import numpy as np

print("Python:", sys.executable)
print("Torch version:", torch.__version__)
print("Transformers ready:", torch.cuda.is_available() or True)

Python: /Users/jeffreyprachick/Library/CloudStorage/OneDrive-Personal/GitHub Projects/review-summarization-pipeline/venv310/bin/python
Torch version: 2.1.2
Transformers ready: True


In [2]:
# Load Fine Food Reviews Dataset
df = pd.read_csv("../data/Reviews.csv")

# Drop rows missing review text or product ID
df = df.dropna(subset=['Text', 'ProductId'])

# Group by product and concatenate the first 10 reviews
df_grouped = df.groupby('ProductId').agg({
    'Text': lambda x: ' '.join(x[:10]),
    'Score': 'mean',
    'Summary': 'first'
}).reset_index()

# Rename for consistency
df_grouped.rename(columns={'Text': 'review_body'}, inplace=True)

# Preview
df_grouped[['ProductId', 'review_body', 'Score']].head()

,ProductId,review_body,Score
0,0006641040,"These days, when a person says, ""chicken soup""...",4.351351
1,141278509X,This product by Archer Farms is the best drink...,5.000000
2,2734888454,My dogs loves this chicken but its a product f...,3.500000
3,2841233731,This book is easy to read and the ingredients ...,5.000000
4,7310172001,This product is a very health snack for your p...,4.751445


In [3]:
# Load PEGASUS Model and Tokenizer
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

model_name = "google/pegasus-xsum"
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)

/Users/jeffreyprachick/Library/CloudStorage/OneDrive-Personal/GitHub Projects/review-summarization-pipeline/venv310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/jeffreyprachick/Library/CloudStorage/OneDrive-Personal/GitHub Projects/review-summarization-pipeline/venv310/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/jeffreyprachick/Library/CloudStorage/OneDrive-Personal/GitHub Projects/review-summarization-pipeline/venv310/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage wil

In [4]:
# Load BART Model and Tokenizer
from transformers import BartTokenizer, BartForConditionalGeneration

bart_model_name = "facebook/bart-large-cnn"
bart_tokenizer = BartTokenizer.from_pretrained(bart_model_name)
bart_model = BartForConditionalGeneration.from_pretrained(bart_model_name)

In [5]:
# Summarize a Single Product's Reviews using PEGASUS
text = df_grouped['review_body'][0]

inputs = tokenizer(
    text,
    truncation=True,
    padding="longest",
    return_tensors="pt",
    max_length=512
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    num_beams=5,
    early_stopping=True
)
pegasus_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original Text:\n", text[:500], "...")
print("\nPEGASUS Summary:\n", pegasus_summary)

Original Text:
 These days, when a person says, "chicken soup" they're probably going to follow up those words with, "for the soul" or maybe "for the teenaged soul".  Didn't used to be that way.  Why I can remember a time when if a person said, "chicken soup" those words were followed by an enthusiastic "with rice!".  Such was the power of Maurice Sendak's catchy 1962 children's book.  I am pleased to report that if you care to read this book again today, you will find it hasn't dimished a jot in terms of froli ...

PEGASUS Summary:
 In our series of letters from African-American journalists, film-maker and columnist Sharon Florentine looks back at some of her favourite books.


In [6]:
# Summarize the Same Review Using BART
bart_inputs = bart_tokenizer(
    text,
    truncation=True,
    padding="longest",
    return_tensors="pt",
    max_length=1024
)

bart_summary_ids = bart_model.generate(
    bart_inputs["input_ids"],
    max_length=60,
    num_beams=4,
    early_stopping=True
)

bart_summary = bart_tokenizer.decode(bart_summary_ids[0], skip_special_tokens=True)

print("BART Summary:\n", bart_summary)

BART Summary:
 Maurice Sendak's 1962 children's book is a classic. Each month gets its own rhythmic poem and accompanying illustration. This book was the "Chicka Chicka Boom Boom" of its day, and still remains the catchiest method to teach kids the months of the year


In [7]:
# Summarize First 10 Products with PEGASUS and BART
from tqdm.auto import tqdm

results = []

# Limit to first 10 products for demo
for i, row in tqdm(df_grouped.head(10).iterrows(), total=10):
    product_id = row["ProductId"]
    review_text = row["review_body"]

    # --- PEGASUS ---
    peg_inputs = tokenizer(
        review_text,
        truncation=True,
        padding="longest",
        return_tensors="pt",
        max_length=512
    )
    peg_ids = model.generate(
        peg_inputs["input_ids"],
        max_length=60,
        num_beams=5,
        early_stopping=True
    )
    pegasus_summary = tokenizer.decode(peg_ids[0], skip_special_tokens=True)

    # --- BART ---
    bart_inputs = bart_tokenizer(
        review_text,
        truncation=True,
        padding="longest",
        return_tensors="pt",
        max_length=1024
    )
    bart_ids = bart_model.generate(
        bart_inputs["input_ids"],
        max_length=60,
        num_beams=4,
        early_stopping=True
    )
    bart_summary = bart_tokenizer.decode(bart_ids[0], skip_special_tokens=True)

    results.append({
        "ProductID": product_id,
        "OriginalText": review_text[:500] + "...",  # Truncated for display
        "PEGASUS_Summary": pegasus_summary,
        "BART_Summary": bart_summary
    })

# Convert to DataFrame
summary_df = pd.DataFrame(results)

# Save to CSV
summary_df.to_csv('/Users/jeffreyprachick/Library/CloudStorage/OneDrive-Personal/GitHub Projects/review-summarization-pipeline/outputs/model_comparison_summaries.csv', index=False)

# Display
summary_df.head()

100%|███████████████████████████████████████████| 10/10 [03:21<00:00, 20.19s/it]


,ProductID,OriginalText,PEGASUS_Summary,BART_Summary
0,0006641040,"These days, when a person says, ""chicken soup""...",In our series of letters from African-American...,Maurice Sendak's 1962 children's book is a cla...
1,141278509X,This product by Archer Farms is the best drink...,Have you ever wondered what it would be like t...,This product by Archer Farms is the best drink...
2,2734888454,My dogs loves this chicken but its a product f...,Is it safe to buy chicken products made in the...,My dogs loves this chicken but its a product f...
3,2841233731,This book is easy to read and the ingredients ...,I have been using this book to make healthy sn...,This book is easy to read and the ingredients ...
4,7310172001,This product is a very health snack for your p...,Our Premium Beef Liver Training Treats are a d...,This product is a very health snack for your p...
